In [21]:
import pickle

with open('shop_dataset.pkl', 'rb') as f:
    shop_data = pickle.load(f)

shop_data.head(3)

,Product_Name,Category,Store_ID,Purchase_Date,Sales_Date,Quantity_Purchased,Quantity_Sold,Purchase_Price,Selling_Price,Discount,Revenue,Stock_Level,Demand_Forecast,Region,Competitor_Price,Marketing_Campaign,Seasonality,Returns
0,Eggs,Snacks,4,2022-01-01 00:00:00.000000000,2022-12-09 21:05:27.272727276,37,1,3.20,12.30,0.14,10.5780,78,16,South,8.19,0,Summer Sales,1
1,Juice,Bakery,6,2022-01-04 16:29:05.454545454,2022-05-21 02:25:27.272727274,45,27,6.07,5.83,0.27,114.9093,9,56,East,17.41,1,Holiday Season,1
2,Cookies,Beverages,7,2022-01-08 08:58:10.909090909,2022-08-13 21:34:32.727272728,14,36,14.71,3.60,0.26,95.9040,6,75,South,12.63,0,Black Friday,1


In [22]:
import pandas as pd
df_encoded = pd.get_dummies(shop_data, columns=['Product_Name', 'Category', 'Region', 'Seasonality'])

features = [
    'Quantity_Purchased', 'Purchase_Price', 'Selling_Price', 'Discount',
    'Stock_Level', 'Competitor_Price', 'Marketing_Campaign', 'Returns',
    'Demand_Forecast'
] + list(df_encoded.columns[df_encoded.columns.str.startswith(('Product_Name_', 'Category_', 'Region_', 'Seasonality_'))])

In [24]:
from sklearn.model_selection import train_test_split
target = 'Demand_Forecast'

X = df_encoded[features]
y = df_encoded[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [25]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(mae_rf)
print(mse_rf)
print(r2_rf)


1.597499999999999
4.610774999999999
0.9947454807048534


In [26]:
from statsmodels.tsa.arima.model import ARIMA
product_data = shop_data[(shop_data['Product_Name'] == 'Product_A') & 
                         (shop_data['Region'] == 'Region_1')]
product_data = product_data.set_index('Sales_Date')
model = ARIMA(product_data['Demand_Forecast'], order=(1, 1, 1))
arima_model = model.fit()
y_pred_arima = arima_model.forecast(steps=len(y_test))
mae_arima = mean_absolute_error(y_test[:len(y_pred_arima)], y_pred_arima)
mse_arima = mean_squared_error(y_test[:len(y_pred_arima)], y_pred_arima)
r2_arima = r2_score(y_test[:len(y_pred_arima)], y_pred_arima)

print(f"ARIMA MAE: {mae_arima}")
print(f"ARIMA MSE: {mse_arima}")
print(f"ARIMA R2: {r2_arima}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWar

LinAlgError: LU decomposition error.

In [27]:
shop_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Product_Name        100 non-null    object        
 1   Category            100 non-null    object        
 2   Store_ID            100 non-null    int32         
 3   Purchase_Date       100 non-null    datetime64[ns]
 4   Sales_Date          100 non-null    datetime64[ns]
 5   Quantity_Purchased  100 non-null    int32         
 6   Quantity_Sold       100 non-null    int32         
 7   Purchase_Price      100 non-null    float64       
 8   Selling_Price       100 non-null    float64       
 9   Discount            100 non-null    float64       
 10  Revenue             100 non-null    float64       
 11  Stock_Level         100 non-null    int32         
 12  Demand_Forecast     100 non-null    int32         
 13  Region              100 non-null    object        


In [28]:
product_data = shop_data[(shop_data['Product_Name'] == 'Product_A') & 
                         (shop_data['Region'] == 'Region_1')]
product_data = product_data.set_index('Sales_Date')  # Установите 'Sales_Date' в качестве индекса


In [29]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Пример вашего DataFrame
# shop_data = pd.read_csv('path_to_your_data.csv')  # Загрузка данных, если нужно

# Используем Sales_Date в качестве индекса
product_data = shop_data[(shop_data['Product_Name'] == 'Product_A') & 
                         (shop_data['Region'] == 'Region_1')]
product_data = product_data.set_index('Sales_Date')

# Обучение ARIMA на данных
model = ARIMA(product_data['Demand_Forecast'], order=(1, 1, 1))
arima_model = model.fit()

# Прогноз на будущее (например, на 3 периода вперед)
forecast = arima_model.forecast(steps=3)
print("ARIMA Forecast:", forecast)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWar

LinAlgError: LU decomposition error.